# 🎯 Government ABAC Demo - Step 1: Create Masking + Filtering Functions

## 📋 Overview
This notebook creates **masking functions** for the Government industry ABAC (Attribute-Based Access Control) demo.

### What are Masking and Filtering Functions?
Masking functions are SQL user-defined functions (UDFs) that transform sensitive data to protect privacy while maintaining data utility for analytics. Row filters let you control which rows a user can access in a table based on custom logic. Masking and filtering functions are the foundation of ABAC policies in Unity Catalog.

### Why Use Masking and Filtering Functions?
- **Compliance**: Meet GDPR, CCPA, HIPAA, and other privacy regulations
- **Security**: Protect sensitive data from unauthorized access
- **Flexibility**: Apply different masks based on user roles and attributes
- **Analytics**: Preserve data utility for analysis while protecting privacy
- **Audit**: Track and log all data access patterns

### What This Notebook Creates
This notebook will create specialized masking functions for the Government industry, including:
- **Identity Protection**: Email, phone, address masking
- **Financial Data**: Credit card, transaction amount bucketing
- **Identifiers**: Deterministic hashing for cross-table analytics
- **Confidential Data**: Complete redaction of sensitive fields
- **Network Data**: IP address masking

## 🎓 How to Use This Notebook
1. **Update Configuration**: Change the catalog name in the configuration cell below
2. **Run All Cells**: Execute cells sequentially (Shift+Enter or Run All)
3. **Verify Success**: Check for ✅ success messages after each function
4. **Proceed to Next Step**: Continue to notebook 2 to create the schema

## ⚙️ Prerequisites
- ✅ Unity Catalog enabled workspace
- ✅ CREATE FUNCTION permission in the target catalog
- ✅ SQL Warehouse or Cluster attached to this notebook
- ✅ Account admin or catalog owner role (recommended)

## 🔄 Next Steps
After completing this notebook:
1. **Step 2**: `2_Create_Tables.ipynb` - Create schema and core tables
2. **Step 3**: `3_Setup_Tagging.ipynb` - Define and apply tags
3. **Step 4**: `4_Test_ABAC_Policies.ipynb` - Test functions through ABAC policies

---


## ⚙️ Configuration

### 🚨 IMPORTANT: Update Before Running!
Change `your_catalog_name` to **your catalog name** in the cell below or update the config.yaml.

### What This Does:
- Sets the target Unity Catalog
- Creates the `government` schema if it doesn't exist

In [0]:
pip install pyyaml

In [0]:
# 📋 Load Configuration from config.yaml
import yaml
from pathlib import Path

config_file = Path('config.yaml')
if config_file.exists():
    with open(config_file) as f:
        config = yaml.safe_load(f)
    CATALOG = config['catalog']
    SCHEMA = config['schema']
    print(f'✅ Configuration loaded from config.yaml')
    print(f'   📊 Catalog: {CATALOG}')
    print(f'   📁 Schema: {SCHEMA}')
else:
    # Fallback defaults
    CATALOG = 'your_catalog_name'
    SCHEMA = 'government'
    print(f'⚠️  config.yaml not found - using defaults')
    print(f'   📊 Catalog: {CATALOG}')
    print(f'   📁 Schema: {SCHEMA}')

# Set catalog and schema to use for the cells below
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"USE SCHEMA {SCHEMA}")


In [0]:
%sql
SELECT '🎯 Target: ' || current_catalog() || '.' || current_schema() AS status;

##COLUMN MASK FUNCTIONS

=============================================

1. SSN MASKING (Last 4 digits)

    Purpose: Mask SSN showing last 4 for identity verification

    Usage: Customer identification

=============================================

In [0]:
%sql
CREATE OR REPLACE FUNCTION mask_ssn_last4(ssn STRING) 
RETURNS STRING
COMMENT 'SSN masking' 
RETURN CASE 
    WHEN ssn IS NULL THEN ssn 
    ELSE CONCAT('XXX-XX-', RIGHT(REPLACE(ssn, '-', ''), 4)) 
END;

=============================================

2. LICENSE MASKING

    Purpose: Partial license number masking for identity verification

    Usage: Analytics without license number exposure

=============================================

In [0]:
%sql
CREATE OR REPLACE FUNCTION mask_license_partial(license STRING) 
RETURNS STRING
RETURN CASE 
  WHEN license IS NULL THEN license 
  ELSE CONCAT('****-', RIGHT(license, 2)) 
END;

=============================================

3. ADDRESS MASKING

    Purpose: Mask address for identity verification

    Usage: User identification

=============================================

In [0]:
%sql
CREATE OR REPLACE FUNCTION mask_address(address STRING) 
RETURNS STRING
RETURN '***';

=============================================

4. AMOUNT BUCKETING (Tax amounts)

    Purpose: Group tax amounts into ranges

    Usage: Analytics without exact amounts

=============================================

In [0]:
%sql
CREATE OR REPLACE FUNCTION mask_tax_amount_bucket(amt DECIMAL(12,2)) 
RETURNS STRING
COMMENT 'Tax ranges' 
RETURN CASE 
  WHEN amt IS NULL THEN 'Unknown' 
  WHEN amt < 10000 THEN '\$0-\$10K'
  WHEN amt < 50000 THEN '\$10K-\$50K' 
  WHEN amt < 100000 THEN '\$50K-\$100K' 
  ELSE '\$100K+' 
END;

=============================================

5. CITIZEN ID MASKING

    Purpose: Hash citizen IDs for privacy

    Usage: Citizen tracking without exposure

=============================================

In [0]:
%sql
CREATE OR REPLACE FUNCTION mask_citizen_id_hash(id STRING) 
RETURNS STRING
COMMENT 'Deterministic' 
RETURN CONCAT('CIT_', SUBSTRING(SHA2(id, 256), 1, 12));

##ROW FILTER FUNCTIONS

=============================================

6. TIME-BASED FILTER: BUSINESS HOURS

    Purpose: Allow data access only during business hours (8AM - 6PM Chicago time)

    Usage: Time-based access control for sensitive data

=============================================


In [0]:
%sql
CREATE OR REPLACE FUNCTION business_hours_filter()
RETURNS BOOLEAN
COMMENT 'ABAC utility: Allow access only during business hours (8AM-6PM America/Chicago)'
RETURN hour(from_utc_timestamp(current_timestamp(), 'America/Chicago')) BETWEEN 8 AND 18;

In [0]:
%sql
SELECT '✅ Government functions created!' AS status;

## ✅ Success!

All Government functions have been created successfully!

### What You Just Created:
- ✅ Masking functions registered in Unity Catalog
- ✅ Functions available for use in SQL queries
- ✅ Foundation for ABAC policies ready

### Verify Your Functions:
You can verify the functions were created by running:
```sql
SHOW FUNCTIONS IN government;
```

### 🎯 Next Step:
Continue to **`2_Create_Tables.ipynb`** to create the database tables and load sample data.

---
**Note**: These functions are stored in Unity Catalog and can be used across multiple notebooks and queries.
